# Notebook 3: Key instance identification

In this notebook, we consider a problem specific to multi-instance learning (MIL): key instance detection (KID). The goal of KID is to identify the instances that are most relevant to the predicted property. For example, in a multi-conformer bioactivity model, KID aims to identify the conformer most strongly associated with the predicted biological activity, often referred to as the bioactive conformer.

In this notebook, we explore how to access instance-weight prediction functionality and how to evaluate KID performance.

In [1]:
import pickle
import random

import numpy as np
import pandas as pd

### 1. Data load

As a dataset for this notebook, we use a manually constructed benchmark in which bioactive conformers are defined as those satisfying predefined pharmacophore patterns (bioactivity triggers). Each bag contains up to 20 conformers per molecule. Active conformers may match different numbers of manually designed pharmacophore templates; consequently, the greater the number of matched pharmacophores, the higher the assigned activity of the conformer. All remaining conformers are considered inactive and do not match any pharmacophore template.

The task is formulated as a regression problem at the bag level. The target value for each bag is defined as the maximum number of matched pharmacophores across all conformers in the bag, ranging from 1 to 7.

In [2]:
from huggingface_hub import hf_hub_download
from qsarmil.utils.ensemble import ConformerEnsemble

/storage/dmitry/miniforge3/envs/milearn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
REPO_ID = "KagakuData/notebooks"

pkl_path = hf_hub_download(REPO_ID, filename="conformer/train_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_train = pickle.load(f)

pkl_path = hf_hub_download(REPO_ID, filename="conformer/test_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_test = pickle.load(f)

In [4]:
# molecules
mols_train = [i[1] for i in data_train]
mols_test = [i[1] for i in data_test]

# conformers
confs_train = [ConformerEnsemble(i) for i in mols_train]
confs_test = [ConformerEnsemble(i) for i in mols_test]

# property
y_train = [i[3] for i in data_train]
y_test = [i[3] for i in data_test]

# active conformers
idx_train = [i[2] for i in data_train]
idx_test = [i[2] for i in data_test]

### 2. Descriptor calculation

Many different types of descriptors can be used to encode conformers; however, the choice of descriptors should satisfy a few key requirements. The descriptors must be able to distinguish between different conformers, meaning that their vector representations should vary sufficiently across conformers. Also, the descriptors should be relevant to the target property, ensuring that they capture chemically meaningful information for the prediction task.

In [5]:
# 3D descriptors
from qsarmil.descriptor.rdkit import RDKitGEOM, RDKitAUTOCORR, RDKitRDF, RDKitMORSE, RDKitWHIM, RDKitGETAWAY
from molfeat.calc import Pharmacophore3D, USRDescriptors, ElectroShapeDescriptors
from qsarmil.descriptor.wrapper import DescriptorWrapper
from milearn.preprocessing import BagMinMaxScaler

In [6]:
desc_calc = DescriptorWrapper(Pharmacophore3D(factory='pmapper'), verbose=True)

In [7]:
# 1. Calculate descriptors
x_train = desc_calc.run(confs_train)
x_test = desc_calc.run(confs_test)

Calculating descriptors: 1317/1317
Calculating descriptors: 356/356


In [8]:
# 2. Scale descriptors
scaler = BagMinMaxScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

### 3. Model building

Not all MIL methods are capable of identifying key instances, and different approaches rely on different mechanisms for KID. One of the most widely used strategies is the attention-based mechanism, where multi-instance neural networks are extended with an attention module that assigns weights to individual instances.

In **QSARmil**, several methods implement this attention-based idea. In addition to standard prediction outputs ``model.predict(x)``, these methods also provide a instance weight prediction ``model.get_instance_weights(x)`` method, which returns a list of weights corresponding to each instance in the input bag.

In [9]:
# Network hparams
from milearn.network.module.hopt import DEFAULT_PARAM_GRID

# MIL networks (with instance weight prediction)
from milearn.network.regressor import (AdditiveAttentionNetworkRegressor, 
                                       SelfAttentionNetworkRegressor,
                                       HopfieldAttentionNetworkRegressor,
                                       DynamicPoolingNetworkRegressor)

In [10]:
model = DynamicPoolingNetworkRegressor()

model.hopt(x_train_scaled, y_train, param_grid=DEFAULT_PARAM_GRID, verbose=True)
model.fit(x_train_scaled, y_train)

Optimizing hyperparameter: activation (5 options)
[1/23 |  4.3% |  0.5 min] Value: relu, Epochs: 40, Loss: 0.0111
[2/23 |  8.7% |  0.5 min] Value: leakyrelu, Epochs: 44, Loss: 0.0122
[3/23 | 13.0% |  0.6 min] Value: gelu, Epochs: 50, Loss: 0.0102
[4/23 | 17.4% |  0.3 min] Value: elu, Epochs: 26, Loss: 0.0318
[5/23 | 21.7% |  0.7 min] Value: silu, Epochs: 88, Loss: 0.0094
Best activation = silu, val_loss = 0.0094
Optimizing hyperparameter: learning_rate (2 options)
[6/23 | 26.1% |  0.5 min] Value: 0.0001, Epochs: 99, Loss: 0.0136
[7/23 | 30.4% |  0.3 min] Value: 0.001, Epochs: 46, Loss: 0.0096
Best learning_rate = 0.001, val_loss = 0.0096
Optimizing hyperparameter: batch_size (3 options)
[8/23 | 34.8% |  0.5 min] Value: 32, Epochs: 43, Loss: 0.0109
[9/23 | 39.1% |  0.4 min] Value: 512, Epochs: 58, Loss: 0.0107
[10/23 | 43.5% |  0.2 min] Value: 1024, Epochs: 25, Loss: 0.0155
Best batch_size = 512, val_loss = 0.0107
Optimizing hyperparameter: weight_decay (5 options)
[11/23 | 47.8% |  0.5

DynamicPoolingNetworkRegressor(
  (instance_transformer): Sequential(
    (0): Linear(in_features=2048, out_features=2048, bias=True)
    (1): SiLU()
    (2): Linear(in_features=2048, out_features=1024, bias=True)
    (3): SiLU()
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): SiLU()
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): SiLU()
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): SiLU()
    (10): Linear(in_features=128, out_features=64, bias=True)
    (11): SiLU()
  )
  (bag_estimator): Norm()
  (dynamic_pooling): DynamicPooling()
)

### 4. KID accuracy evaluation

To evaluate **key instance detection (KID)** performance, we use a function that compares predicted instance importance scores with ground-truth key instance annotations.

The inputs are structured as follows: ``y_true`` is a binary indicator vector where each element corresponds to an instance in the bag and active bits are the true key instances. ``y_pred`` is a vector of the same shape, where each element represents the predicted weight assigned to the corresponding instance.

The evaluation returns two metrics. The first is the empirical KID accuracy, which measures how often the top-ranked predicted instances correctly identify at least one true key instance. The second is the expected KID accuracy, which serves as a random-selection baseline and reflects the probability of hitting a key instance by chance given the number of positives and the bag size.

In [11]:
# KID accuracy metrics
from sklearn.metrics import r2_score
from qsarmil.utils.metrics import kid_accuracy

In [12]:
# convert indeces to binary vector
def idx_to_binary(x, k):
    y = []
    for bag, ki in zip(x, k):
        n = len(bag)
        label = np.zeros(n, dtype=int)
        label[ki] = 1
        y.append(label)
    return y

In [13]:
# key binary 
keys_train = idx_to_binary(confs_train, idx_train)
keys_test = idx_to_binary(confs_test, idx_test)

In [14]:
y_pred = model.predict(x_test_scaled)
w_pred = model.get_instance_weights(x_test_scaled)
w_pred = [w.flatten() for w in w_pred]

In [15]:
top_n = 1

print(f"All molecules: {len(y_test)}")
print(f"Prediction accuracy: {r2_score(y_test, y_pred):.2f}")

acc, exp = kid_accuracy(keys_test, w_pred, top_n=top_n)
print(f"KID prediction accuracy: {acc:.2f}")
print(f"KID baseline accuracy: {exp:.2f}")

idx_7 = []
for n, y in enumerate(y_test):
    if y == 7:
        idx_7.append(n)
keys_test_7 = [keys_test[i] for i in idx_7]
w_pred_7 = [w_pred[i] for i in idx_7]

print(f"\nActive molecules: {len(idx_7)}")

acc, exp = kid_accuracy(keys_test_7, w_pred_7, top_n=top_n)
print(f"KID prediction accuracy: {acc:.2f}")
print(f"KID baseline accuracy: {exp:.2f}")

All molecules: 356
Prediction accuracy: 0.91
KID prediction accuracy: 0.24
KID baseline accuracy: 0.11

Active molecules: 66
KID prediction accuracy: 0.45
KID baseline accuracy: 0.16


### 5. KID prediction visualization

The conformers and their corresponding predicted instance weights can be visually inspected using the utility function provided below. This allows manual exploration of how the model assigns importance scores across different conformations, providing an intuitive way to interpret key instance detection results.

In [16]:
from qsarmil.utils.visualization import visualize_conformers_grid

In [20]:
# most active molecules indeces
print(idx_7)

[3, 9, 15, 16, 29, 39, 45, 47, 48, 52, 55, 56, 63, 77, 83, 95, 96, 98, 100, 101, 102, 104, 114, 120, 132, 156, 158, 173, 174, 179, 181, 197, 210, 211, 212, 213, 214, 215, 216, 224, 228, 231, 247, 252, 253, 255, 256, 269, 271, 276, 281, 284, 285, 294, 296, 297, 300, 304, 308, 314, 336, 346, 349, 351, 352, 353]


In [18]:
N = 45 # choose molecule index from the list above

print(y_pred[N])
print(y_test[N])

6.1239014
7


In [19]:
visualize_conformers_grid(mols_test[N], w_pred[N], idx_test[N], top_n=5, sort_by_weight=True)